# Pruebas de deduplicación y record linkage — `record_linkage`

Cuaderno de referencia para medir el desempeño del motor sobre el **Ground
Truth Multicampo**. Cubre los dos modos de uso y deja métricas comparables
entre versiones (precisión, recall, F1 a nivel de pares, y desglose por caso).

**Cómo usarlo**
1. Ejecute la **Sección 0** (montaje, rutas, instalación) una vez.
2. Ejecute la **Sección 1 (EXTRAS)** una vez: carga toda la lógica.
3. Corra las celdas **EJECUTAR** de deduplicación (Sección 2) y de record
   linkage (Sección 3). Cada una imprime sus métricas.
4. La **Sección 4** guarda un reporte de métricas con fecha para comparar a futuro.

> Arquitectura: la lógica vive en celdas EXTRAS; las celdas EJECUTAR solo
> tienen configuración. No hay números mágicos escondidos.


## Sección 0 · Configuración del entorno

In [1]:
# ── Montaje de Google Drive (solo en Colab) ────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
    print("Fuera de Colab: ajuste las rutas manualmente si es necesario.")

Mounted at /content/drive


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ► ÚNICO lugar que modifica: rutas del paquete y de los datos.     ║
# ╚══════════════════════════════════════════════════════════════════╝
RUTA_PAQUETE = "/content/drive/MyDrive/ProColombia/record_linkage"
DIR_DATOS    = "/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-07-13"
ARCHIVO_GT   = "Ground_Truth_Multicampo_v1.xlsx"

In [3]:
# ── Instalación del paquete ────────────────────────────────────────────────
# NOTA: el paquete usa layout src/ (src/record_linkage). En Colab, un
# `pip install -e` crea un finder editable en un archivo .pth que SOLO se lee
# al arrancar el intérprete; como el kernel ya está corriendo, ese finder no
# se activa y `import record_linkage` falla con ModuleNotFoundError. Por eso,
# además de instalar (para traer las dependencias), añadimos src/ a sys.path
# de forma explícita: así el import funciona en ESTA sesión sin reiniciar.
import importlib
import subprocess
import sys
from pathlib import Path

ruta_pkg = Path(RUTA_PAQUETE)
src_dir = ruta_pkg / "src"

if ruta_pkg.exists():
    # 1) Instalar dependencias declaradas (pandas, numpy, rapidfuzz, datasketch, ...)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ruta_pkg), "-q"],
                   check=True)
    # 2) Garantizar el import en esta sesión (el .pth editable no se relee)
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    print("✅ Paquete instalado desde", ruta_pkg)
    if not src_dir.exists():
        raise FileNotFoundError(
            f"No existe {src_dir}. Verifique que subió el repo completo "
            f"(con la carpeta src/) a:\n   {ruta_pkg}"
        )
else:
    # Fallback: instalar desde PyPI si el paquete ya fue publicado.
    subprocess.run([sys.executable, "-m", "pip", "install", "rues-linker", "-q"], check=True)
    print("⚠️  Ruta local no encontrada; instalado 'rues-linker' desde PyPI.")

# Refrescar caches de importación por si una versión previa quedó cargada.
importlib.invalidate_caches()
if "record_linkage" in sys.modules:
    importlib.reload(sys.modules["record_linkage"])
import record_linkage as rl

print("Versión del motor:", rl.__version__)
print("Importado desde  :", rl.__file__)

✅ Paquete instalado desde /content/drive/MyDrive/ProColombia/record_linkage
Versión del motor: 0.11.0
Importado desde  : /content/drive/MyDrive/ProColombia/record_linkage/src/record_linkage/__init__.py


## Sección 1 · EXTRAS / UTILIDADES (ejecutar una vez)

Toda la lógica reutilizable: configuración, carga del ground truth, esquemas,
cálculo de métricas y desglose por caso. Las celdas EJECUTAR de abajo solo
llaman a estas funciones.


In [4]:
# ===== SECCIÓN EXTRAS / UTILIDADES =====
from __future__ import annotations

from dataclasses import dataclass, field
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from record_linkage.matching.campos import (
    CampoSpec, EsquemaCampos, PoliticaFaltante, TipoCampo,
    esquema_multicampo_completo,
)
from record_linkage.matching.motor_bloqueo import (
    BloqueoComponible, LlaveExacta, LSHTexto, RejillaGeo,
)
from record_linkage.matching.motor_multicampo import (
    evaluar_esquema, clusters_desde_decisiones,
)
from record_linkage.matching.normalizadores import normalizar_campo


@dataclass
class ConfigPrueba:
    """Parámetros de una corrida de prueba. Validación en __init__."""
    umbral_lsh: float = 0.35
    permutaciones: int = 64
    ngram: int = 3
    respetar_vetos: bool = True
    columnas_texto: tuple[str, ...] = ("NIT", "EMAIL", "TELEFONO", "DIRECCION")

    def __post_init__(self):
        if not 0 < self.umbral_lsh < 1:
            raise ValueError(f"umbral_lsh={self.umbral_lsh} fuera de (0, 1)")
        if self.permutaciones < 16:
            raise ValueError("permutaciones muy bajo (mínimo 16)")

In [5]:
def cargar_ground_truth(dir_datos: str, archivo: str) -> dict[str, pd.DataFrame]:
    """Carga todas las hojas del Excel de ground truth. Fail fast si no existe.

    Devuelve un dict {nombre_hoja: DataFrame}. Normaliza tipos: geo a float,
    columnas de texto con NaN → '' (Excel deja celdas vacías como NaN, que el
    motor debe ver como faltante '').
    """
    ruta = Path(dir_datos) / archivo
    if not ruta.exists():
        raise FileNotFoundError(
            f"❌ No se encontró el ground truth en:\n   {ruta}\n"
            f"   Verifique DIR_DATOS y que el archivo esté subido a Drive."
        )
    hojas = pd.read_excel(ruta, sheet_name=None)
    for nombre, df in hojas.items():
        for c in ("LATITUD", "LONGITUD", "LAT", "LON"):
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        for c in ("NIT", "EMAIL", "TELEFONO", "DIRECCION", "RAZON_SOCIAL", "CIUDAD"):
            if c in df.columns:
                df[c] = df[c].fillna("").astype(str)
    print(f"✅ Ground truth cargado: {len(hojas)} hojas desde {ruta.name}")
    for nombre, df in hojas.items():
        print(f"   • {nombre:24s} {df.shape[0]:>4} filas × {df.shape[1]} columnas")
    return hojas

In [6]:
def _pares_de_grupos(labels: np.ndarray) -> set[tuple[int, int]]:
    """Conjunto de pares (i<j) que comparten etiqueta de grupo."""
    s = pd.Series(labels)
    out: set[tuple[int, int]] = set()
    for _, idx in s.groupby(s).groups.items():
        a = sorted(np.asarray(idx).tolist())
        if len(a) > 1:
            out.update(combinations(a, 2))
    return out


def metricas_pares(verdad: np.ndarray, predicho: np.ndarray) -> dict:
    """Precisión, recall y F1 a nivel de pares entre dos etiquetados."""
    V = _pares_de_grupos(verdad)
    P = _pares_de_grupos(predicho)
    tp, fp, fn = len(P & V), len(P - V), len(V - P)
    prec = tp / (tp + fp) if tp + fp else 1.0
    rec = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4),
            "tp": tp, "fp": fp, "fn": fn,
            "grupos_predichos": int(len(np.unique(predicho))),
            "grupos_verdaderos": int(len(np.unique(verdad)))}


def imprimir_metricas(titulo: str, m: dict) -> None:
    print(f"""
📊 {titulo}
   Precisión : {m['precision']:.4f}
   Recall    : {m['recall']:.4f}
   F1        : {m['f1']:.4f}
   TP={m['tp']}  FP={m['fp']}  FN={m['fn']}
   Grupos: {m['grupos_predichos']} predichos vs {m['grupos_verdaderos']} verdaderos
""")

In [7]:
def bloqueo_estandar(cfg: ConfigPrueba) -> BloqueoComponible:
    """Bloqueo recomendado: llaves exactas (NIT/email/teléfono) + LSH de nombre.

    No incluye rejilla geo a propósito: sobre este dataset la geo como bloqueo
    añade candidatos sin mejorar el recall (la medición está documentada en el
    CHANGELOG del motor). El tipo GEO sí puede usarse dentro del esquema de score.
    """
    return BloqueoComponible([
        LlaveExacta("NIT"),
        LlaveExacta("EMAIL"),
        LlaveExacta("TELEFONO"),
        LSHTexto("RAZON_SOCIAL", umbral=cfg.umbral_lsh,
                 permutaciones=cfg.permutaciones, ngram=cfg.ngram),
    ])


def deduplicar(df: pd.DataFrame, esquema: EsquemaCampos,
               cfg: ConfigPrueba | None = None) -> tuple[np.ndarray, object]:
    """Ejecuta el pipeline de deduplicación y devuelve (etiquetas, resultado)."""
    cfg = cfg or ConfigPrueba()
    bloq = bloqueo_estandar(cfg)
    res = evaluar_esquema(df, esquema, bloq)
    labels = clusters_desde_decisiones(len(df), res.decisiones,
                                       respetar_vetos=cfg.respetar_vetos)
    print(f"✅ Dedup: {res.n_candidatos} candidatos → {res.n_fusiones} fusiones "
          f"→ {len(np.unique(labels))} grupos")
    return labels, res

In [8]:
def enlazar_bases(df_a: pd.DataFrame, df_b: pd.DataFrame, esquema: EsquemaCampos,
                  cfg: ConfigPrueba | None = None
                  ) -> tuple[pd.DataFrame, set[tuple[int, int]]]:
    """Record linkage entre dos bases. Devuelve (union_con_cluster, pares_AB).

    Une A y B, clusteriza, y extrae los pares (REG_ID_A, REG_ID_B) que caen en
    el mismo grupo. Requiere que A tenga 'REG_ID_A' y B tenga 'REG_ID_B'.
    """
    cfg = cfg or ConfigPrueba()
    a = df_a.rename(columns={"REG_ID_A": "REG_ID"}).copy(); a["ORIGEN"] = "A"
    b = df_b.rename(columns={"REG_ID_B": "REG_ID"}).copy(); b["ORIGEN"] = "B"
    a["GID"] = range(len(a))
    b["GID"] = range(len(a), len(a) + len(b))
    union = pd.concat([a, b], ignore_index=True)

    bloq = bloqueo_estandar(cfg)
    res = evaluar_esquema(union, esquema, bloq)
    union["CLUSTER"] = clusters_desde_decisiones(len(union), res.decisiones,
                                                 respetar_vetos=cfg.respetar_vetos)
    g2o = dict(zip(union["GID"], union["ORIGEN"]))
    g2r = dict(zip(union["GID"], union["REG_ID"]))
    pares_ab: set[tuple[int, int]] = set()
    for _, grp in union.groupby("CLUSTER"):
        gids = grp["GID"].tolist()
        for i in range(len(gids)):
            for j in range(i + 1, len(gids)):
                gi, gj = gids[i], gids[j]
                if g2o[gi] != g2o[gj]:
                    ra = g2r[gi] if g2o[gi] == "A" else g2r[gj]
                    rb = g2r[gj] if g2o[gj] == "B" else g2r[gi]
                    pares_ab.add((int(ra), int(rb)))
    print(f"✅ Linkage: {res.n_candidatos} candidatos → {len(pares_ab)} cruces A↔B")
    return union, pares_ab


def metricas_linkage(pares_pred: set[tuple[int, int]],
                     gt: pd.DataFrame) -> dict:
    """Métricas de cruce A↔B contra el ground truth de linkage."""
    verdad = set(zip(gt["REG_ID_A"].astype(int), gt["REG_ID_B"].astype(int)))
    tp, fp, fn = len(pares_pred & verdad), len(pares_pred - verdad), len(verdad - pares_pred)
    prec = tp / (tp + fp) if tp + fp else 1.0
    rec = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4),
            "tp": tp, "fp": fp, "fn": fn,
            "matches_predichos": len(pares_pred), "matches_verdaderos": len(verdad)}

In [9]:
def desglose_por_caso(df: pd.DataFrame, labels: np.ndarray,
                      col_grupo: str = "ID_GRUPO_ESPERADO",
                      col_caso: str = "CASO_TIPO") -> pd.DataFrame:
    """Precisión/recall por tipo de caso: dónde acierta y dónde falla el motor.

    Para cada caso cuenta pares positivos correctamente fusionados (recall) y
    revisa si hubo fusiones indebidas que involucren registros de ese caso.
    """
    df = df.reset_index(drop=True)
    V = _pares_de_grupos(df[col_grupo].to_numpy())
    P = _pares_de_grupos(labels)
    filas = []
    for caso in sorted(df[col_caso].dropna().unique()):
        idx_caso = set(df.index[df[col_caso] == caso])
        v_caso = {(a, b) for a, b in V if a in idx_caso or b in idx_caso}
        if not v_caso:
            # caso negativo (sin pares positivos): medir fusiones indebidas
            fp_caso = {(a, b) for a, b in (P - V) if a in idx_caso and b in idx_caso}
            filas.append({"CASO": caso, "tipo": "negativo/edge",
                          "pares_pos": 0, "recuperados": 0, "recall": None,
                          "fusiones_indebidas": len(fp_caso)})
        else:
            rec = len(v_caso & P) / len(v_caso)
            filas.append({"CASO": caso, "tipo": "positivo",
                          "pares_pos": len(v_caso), "recuperados": len(v_caso & P),
                          "recall": round(rec, 3), "fusiones_indebidas": 0})
    return pd.DataFrame(filas)

## Sección 2 · Deduplicación (duplicados dentro de UNA base)

Detecta y agrupa registros que son la misma entidad dentro de la hoja
`DEDUP_REGISTROS`. La verdad está en `ID_GRUPO_ESPERADO`.


In [10]:
# ── EJECUTAR · cargar el ground truth ──────────────────────────────────────
if "GT" in globals():
    del GT
    import gc; gc.collect()

GT = cargar_ground_truth(DIR_DATOS, ARCHIVO_GT)
dedup = GT["DEDUP_REGISTROS"]

✅ Ground truth cargado: 8 hojas desde Ground_Truth_Multicampo_v1.xlsx
   • LEEME                      37 filas × 1 columnas
   • DEDUP_REGISTROS           166 filas × 21 columnas
   • DEDUP_PARES               140 filas × 6 columnas
   • LINKAGE_BASE_A             20 filas × 16 columnas
   • LINKAGE_BASE_B             21 filas × 16 columnas
   • LINKAGE_GROUND_TRUTH       15 filas × 7 columnas
   • CATALOGO_CASOS             43 filas × 5 columnas
   • DICCIONARIO                22 filas × 5 columnas


In [11]:
# ── EJECUTAR · deduplicación con el esquema de referencia ──────────────────
CFG = ConfigPrueba(umbral_lsh=0.35, respetar_vetos=True)
ESQUEMA = esquema_multicampo_completo()   # NIT, RAZON_SOCIAL, TELEFONO, EMAIL, DIRECCION, CIUDAD

labels_dedup, res_dedup = deduplicar(dedup, ESQUEMA, CFG)
dedup["ID_GRUPO_PREDICHO"] = labels_dedup

m_dedup = metricas_pares(dedup["ID_GRUPO_ESPERADO"].to_numpy(), labels_dedup)
imprimir_metricas("Deduplicación (esquema de referencia)", m_dedup)

✅ Dedup: 513 candidatos → 129 fusiones → 91 grupos

📊 Deduplicación (esquema de referencia)
   Precisión : 1.0000
   Recall    : 0.9847
   F1        : 0.9923
   TP=129  FP=0  FN=2
   Grupos: 91 predichos vs 89 verdaderos



### 2.1 · Desglose por caso — dónde acierta y dónde falla

El recall por caso muestra qué tipos de variación resuelve el motor. Los casos
`C09` y `C21` (NITs distintos para la misma entidad) son **fronteras conocidas**:
el veto de NIT los separa hoy; la Fase 3 debe unirlos con evidencia multicampo.


In [12]:
# ── EJECUTAR · desglose por caso ───────────────────────────────────────────
desglose = desglose_por_caso(dedup, labels_dedup)
print("Recall por caso (positivos) y fusiones indebidas (negativos):")
desglose

Recall por caso (positivos) y fusiones indebidas (negativos):


,CASO,tipo,pares_pos,recuperados,recall,fusiones_indebidas
0,C01,positivo,3,3,1.0,0
1,C02,positivo,6,6,1.0,0
2,C03,positivo,6,6,1.0,0
3,C04,positivo,3,3,1.0,0
4,C05,positivo,3,3,1.0,0
5,C06,positivo,1,1,1.0,0
6,C07,positivo,1,1,1.0,0
7,C08,positivo,1,1,1.0,0
8,C09,positivo,1,0,0.0,0
9,C11,negativo/edge,0,0,NaN,0


### 2.2 · Esquema personalizado (opcional)

Puede declarar su propio esquema: elija columnas, tipos, pesos y política de
faltantes. Este ejemplo añade `SECTOR` (categórico) y `FECHA_CONSTITUCION`
(fecha) al esquema. Compare las métricas con las de referencia.


In [13]:
# ── EJECUTAR · esquema personalizado con más variables ─────────────────────
ESQUEMA_PLUS = EsquemaCampos(
    campos=[
        CampoSpec("NIT", TipoCampo.IDENTIFICADOR, peso=3.0),
        CampoSpec("RAZON_SOCIAL", TipoCampo.NOMBRE_EMPRESA, peso=2.0),
        CampoSpec("EMAIL", TipoCampo.EMAIL, peso=1.5),
        CampoSpec("TELEFONO", TipoCampo.TELEFONO, peso=1.5),
        CampoSpec("CIUDAD", TipoCampo.CIUDAD, peso=0.5),
        CampoSpec("SECTOR", TipoCampo.CATEGORICO, peso=0.5),
        CampoSpec("FECHA_CONSTITUCION", TipoCampo.FECHA, peso=0.5,
                  params={"dias_tolerancia": 15}),
    ],
    umbral_score=0.60, min_concordancias=3, nombre="dedup_plus",
)
labels_plus, _ = deduplicar(dedup, ESQUEMA_PLUS, CFG)
imprimir_metricas("Deduplicación (esquema PLUS: +sector +fecha)",
                  metricas_pares(dedup["ID_GRUPO_ESPERADO"].to_numpy(), labels_plus))

✅ Dedup: 513 candidatos → 129 fusiones → 91 grupos

📊 Deduplicación (esquema PLUS: +sector +fecha)
   Precisión : 1.0000
   Recall    : 0.9847
   F1        : 0.9923
   TP=129  FP=0  FN=2
   Grupos: 91 predichos vs 89 verdaderos



### 2.3 · Veto de NIT condicional — corroboración (Fase 3, v0.11.0)

Por defecto, dos registros con **NITs distintos** nunca se fusionan (el veto de
identificador los separa). La Fase 3 permite *levantar* ese veto cuando hay
**evidencia independiente fuerte**: email y/o teléfono idénticos (valores de alta
entropía) más un nombre muy similar. Las trampas del ground truth (email genérico
gmail compartido, teléfono de call center) siguen rechazándose.

Compare el recall con y sin corroboración: debe recuperar los casos `C09` (NIT con
un dígito errado) y `C21` (empresa con dos NITs) llevando el recall a 1.0000 **sin
perder precisión**.


In [14]:
# ── EJECUTAR · activar la corroboración del veto (F3) ──────────────────────
from record_linkage import CorroboracionVeto

ESQUEMA_F3 = esquema_multicampo_completo()
ESQUEMA_F3.corroboracion = CorroboracionVeto(
    campos_corroborantes=("EMAIL", "TELEFONO"),  # evidencia de alta entropía
    umbral_campo=0.99,            # "idéntico" = casi exacto
    min_corroborantes=1,          # basta email O teléfono idéntico
    umbral_nombre_empresa=0.90,   # además el nombre debe ser muy similar
    activa=True,
)
labels_f3, res_f3 = deduplicar(dedup, ESQUEMA_F3, CFG)
m_f3 = metricas_pares(dedup["ID_GRUPO_ESPERADO"].to_numpy(), labels_f3)
imprimir_metricas("Deduplicación (F3: veto condicional ON)", m_f3)
print(f"Vetos levantados por corroboración: {int(res_f3.decisiones['veto_levantado'].sum())}")

✅ Dedup: 513 candidatos → 131 fusiones → 89 grupos

📊 Deduplicación (F3: veto condicional ON)
   Precisión : 1.0000
   Recall    : 1.0000
   F1        : 1.0000
   TP=131  FP=0  FN=0
   Grupos: 89 predichos vs 89 verdaderos

Vetos levantados por corroboración: 3


## Sección 3 · Record linkage (cruce entre DOS bases)

Cruza `LINKAGE_BASE_A` (CRM) con `LINKAGE_BASE_B` (RUES). La verdad está en
`LINKAGE_GROUND_TRUTH` (pares A↔B que son la misma empresa). Incluye trampas:
homónimos cross-base y entidades presentes en una sola base.


In [15]:
# ── EJECUTAR · cargar las bases A y B ──────────────────────────────────────
base_a = GT["LINKAGE_BASE_A"]
base_b = GT["LINKAGE_BASE_B"]
gt_linkage = GT["LINKAGE_GROUND_TRUTH"]
print(f"Base A (CRM): {len(base_a)} · Base B (RUES): {len(base_b)} · "
      f"matches verdaderos: {len(gt_linkage)}")

Base A (CRM): 20 · Base B (RUES): 21 · matches verdaderos: 15


In [16]:
# ── EJECUTAR · record linkage A ↔ B ────────────────────────────────────────
union_ab, pares_ab = enlazar_bases(base_a, base_b, ESQUEMA, CFG)

m_link = metricas_linkage(pares_ab, gt_linkage)
imprimir_metricas("Record linkage A↔B", {**m_link,
    "grupos_predichos": m_link["matches_predichos"],
    "grupos_verdaderos": m_link["matches_verdaderos"]})

✅ Linkage: 16 candidatos → 15 cruces A↔B

📊 Record linkage A↔B
   Precisión : 1.0000
   Recall    : 1.0000
   F1        : 1.0000
   TP=15  FP=0  FN=0
   Grupos: 15 predichos vs 15 verdaderos



### 3.1 · Revisar los cruces encontrados

Inspeccione qué registros de A se cruzaron con cuáles de B, y compare con el
ground truth. Útil para entender falsos positivos/negativos si los hubiera.


In [17]:
# ── EJECUTAR · tabla de cruces predichos vs verdad ─────────────────────────
verdad_set = set(zip(gt_linkage["REG_ID_A"].astype(int),
                     gt_linkage["REG_ID_B"].astype(int)))
filas_cruce = [{"REG_A": ra, "REG_B": rb,
                "correcto": (ra, rb) in verdad_set} for ra, rb in sorted(pares_ab)]
faltantes = [{"REG_A": ra, "REG_B": rb, "correcto": "NO DETECTADO"}
             for ra, rb in sorted(verdad_set - pares_ab)]
pd.DataFrame(filas_cruce + faltantes)

,REG_A,REG_B,correcto
0,1,1,True
1,2,2,True
2,3,3,True
3,4,4,True
4,5,5,True
5,6,6,True
6,7,7,True
7,8,8,True
8,9,9,True
9,10,10,True


## Sección 4 · Guardar reporte de métricas (para comparar a futuro)

Guarda un JSON con fecha, versión del motor y todas las métricas. Ejecútelo tras
cada mejora del motor y conserve los archivos: la evolución de F1 entre fechas
es la evidencia del avance.


In [18]:
# ── EJECUTAR · guardar reporte con fecha ───────────────────────────────────
import json
from datetime import datetime, timezone

reporte = {
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "version_motor": rl.__version__,
    "ground_truth": ARCHIVO_GT,
    "config": {"umbral_lsh": CFG.umbral_lsh, "respetar_vetos": CFG.respetar_vetos},
    "deduplicacion": m_dedup,
    "record_linkage": m_link,
}
ruta_rep = Path(DIR_DATOS) / f"reporte_metricas_{rl.__version__}_{datetime.now():%Y%m%d}.json"
ruta_rep.write_text(json.dumps(reporte, indent=2, ensure_ascii=False))
print("📋 Reporte guardado en:", ruta_rep)
print(json.dumps(reporte, indent=2, ensure_ascii=False))

📋 Reporte guardado en: /content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-07-13/reporte_metricas_0.11.0_20260715.json
{
  "fecha_utc": "2026-07-15T01:16:22.555716+00:00",
  "version_motor": "0.11.0",
  "ground_truth": "Ground_Truth_Multicampo_v1.xlsx",
  "config": {
    "umbral_lsh": 0.35,
    "respetar_vetos": true
  },
  "deduplicacion": {
    "precision": 1.0,
    "recall": 0.9847,
    "f1": 0.9923,
    "tp": 129,
    "fp": 0,
    "fn": 2,
    "grupos_predichos": 91,
    "grupos_verdaderos": 89
  },
  "record_linkage": {
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "tp": 15,
    "fp": 0,
    "fn": 0,
    "matches_predichos": 15,
    "matches_verdaderos": 15
  }
}


---
### Líneas base esperadas (motor v0.11.0)

Con el ground truth v1 y el esquema de referencia, úselas como umbral de
no-regresión:

- **Deduplicación (veto estándar, corroboración OFF):** P = 1.0000 · R = 0.9847 ·
  F1 = 0.9923 (2 falsos negativos esperados: C09 y C21, por el veto de NIT).
- **Deduplicación (corroboración F3 ON):** P = 1.0000 · R = 1.0000 · F1 = 1.0000 —
  recupera C09 y C21 sin introducir falsos positivos.
- **Record linkage:** P = 1.0000 · R = 1.0000 · F1 = 1.0000.

Una mejora del motor debe **mantener o subir** estas cifras. La corroboración
está **inactiva por defecto**: actívela explícitamente (sección 2.3) para el
comportamiento F3.
